Create a csv file with the county FIPS code and the following county-level census data:
- 2022 Land area in square kilometers
- 2022 Water area in square kilometers
- 2022 Farm area in square kilometers
- 2022 Crop area in square kilometers
- 2020 Population
- 2023 RUCC scores

In [ ]:
# geopandas not included in erdos environment, need to download
# Download Census TIGER/Line shapefules here: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html

import geopandas as gpd
import pandas as pd

counties = gpd.read_file('../../work/tl_2022_us_county/tl_2022_us_county.shp')


In [ ]:
counties = counties.rename(
    columns = {
        'GEOID': 'FIPS',
    }
)

counties.head()

In [ ]:
RUCC_df = pd.read_excel('data/Ruralurbancontinuumcodes2023.xlsx', dtype={'FIPS': str})

RUCC_df.head()

In [ ]:
xls = pd.ExcelFile('data/NASSAgcensusDownload2022.xlsx')

crop_df = pd.read_excel(xls, 'Farms', dtype={'FIPSTEXT': str})

crop_df['FIPSTEXT']


In [ ]:
desired_columns = ['FIPSTEXT','y22_M050_valueNumeric', 'y22_M052_valueNumeric']

crop_df_acres = crop_df[desired_columns].rename(
    columns = {
        'FIPSTEXT': 'FIPS',
        'y22_M050_valueNumeric': 'farm_acre_as_percent',
        'y22_M052_valueNumeric': 'crop_acre_as_percent'
    }
)

crop_df_acres.head()

In [ ]:
all_data = pd.merge(counties, crop_df_acres, on='FIPS')
all_data = pd.merge(all_data, RUCC_df, on='FIPS')

all_data.head()

In [ ]:
county_area = all_data[["FIPS", "ALAND", 'AWATER', 'farm_acre_as_percent', 'crop_acre_as_percent', 'Population_2020', 'RUCC_2023']].copy()

county_area["area_land_km2"] = (
    county_area["ALAND"] / 1_000_000
)

county_area["area_water_km2"] = (
    county_area["AWATER"] / 1_000_000
)

county_area['farm_area_km2'] = (
    county_area['area_land_km2'] * county_area['farm_acre_as_percent'] / 100
)

county_area['crop_area_km2'] = (
    county_area['area_land_km2'] * county_area['crop_acre_as_percent'] / 100
)




In [ ]:
county_area.sample(10)

In [ ]:
county_area.to_csv("../data/county_data_2022.csv", index=False)